In [1]:
import openai
from openai import OpenAI

from tqdm.auto import tqdm
import time
import os
import sys


with open("../../../openai_api_key.sh", "r") as f:
    line = f.readline()
    env_var_name, api_key = line.split("=")

/home/users2/vaethdk/.virtualenvs/cts_al/lib64/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.append('../../..')
print(os.path.realpath("."))

/mount/arbeitsdaten41/projekte/asr-2/vaethdk/cts_activelearning/conversational-tree-search/generation/reimburse/gpt4o


In [3]:
from data.dataset import ReimburseGraphDataset, StandardGraphDataset, DataAugmentationLevel, NodeType, DialogNode, Question

In [4]:

reimburse_human_data = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')

===== Dataset Statistics =====
- files:  en/reimburse/train_graph.json en/reimburse/train_answers.json
- synonyms: True
- depth: 20  - degree: 13
- answers: 312
- questions: 279
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  7
- answer limit: 0  - maximum loaded:  9


In [8]:
system = """You are a helpful assistant creating a list of FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Remove some information, especially nouns and named entities, between generated questions.
Use casual language.
Output only the generated paraphrases, separating each paraphrase with a <br> tag."""

NUM_QUESTIONS = 100
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 15000
generated_data = {}


user_text = """Generate 100 FAQ-style questions from the fact: "In the US, you are entitled to 30$ per day, minus any free meals which you choose to decline."""
messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": user_text},
]

In [5]:
client = OpenAI(api_key=api_key)

# output = client.chat.completions.create(
#         model="gpt-4o",
#         messages=messages,
#         temperature=TEMPERATURE,
#         stream=False,
#     )
# print(output)

In [36]:
# output.choices[0].message.content

In [6]:
def generate_prompt(system: str, user: str):
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

In [18]:
def generate_output(prompt, temperature: float, seed: int) -> str:
    output = client.chat.completions.create(
        model="gpt-4o",
        messages=prompt,
        temperature=temperature,
        stream=False,
        seed=seed
    )
    return output.choices[0].message.content
    

In [19]:
def parse_output(result):
    uniques = set()
    results = []
    duplicates = 0
    for split in result.split("<br>"):
        cleaned = split.strip().strip("\n")
        if len(cleaned.split(" ")) < 3:
            print("SHORT:", cleaned)
        elif len(cleaned.split(" ")) > 100:
            print("LONG:", cleaned)
        else:
            if not cleaned.lower() in uniques:
                results.append(cleaned.strip())
                uniques.add(cleaned.lower())
            else:
                duplicates += 1
    print(" - duplicates:", duplicates)
    return results

# V1

In [58]:
from data.dataset import NodeType, Question
import time

SEED = 42


system = """You are a helpful assistant creating a list of FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Remove some information, especially nouns and named entities, between generated questions.
Use casual language.
Output only the generated paraphrases, separating each paraphrase with a <br> tag."""

def user(answer_text: str, num_paraphrases: int) -> str:
    return f'Generate {num_paraphrases} FAQ-style questions from the fact: "{answer_text}"'

NUM_QUESTIONS = 100
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 15000
SEED = 42
generated_data = {}

for node in tqdm(reimburse_human_data.nodes_by_type[NodeType.INFO]):
    prompt = generate_prompt(system=system, user=user(node.text, NUM_QUESTIONS))
    candidates = generate_output(prompt=prompt, temperature=TEMPERATURE, seed=SEED)
    for candidate in parse_output(candidates):
        key = str(time.time()).replace(".", "")
        generated_data[key] = {
            "dialog_node_key": node.key,
            "key": key,
            "text": candidate,
        }

  1%|▏         | 1/80 [00:24<32:09, 24.42s/it]

SHORT: 
 - duplicates: 2


  2%|▎         | 2/80 [00:44<28:14, 21.72s/it]

SHORT: 
 - duplicates: 9


  4%|▍         | 3/80 [01:20<36:29, 28.43s/it]

...

SHORT: 
 - duplicates: 6


 99%|█████████▉| 79/80 [33:54<00:22, 22.06s/it]

 - duplicates: 2


100%|██████████| 80/80 [34:19<00:00, 25.74s/it]

SHORT: 
 - duplicates: 0


In [59]:
import json

cleaned_data = {}
for key in generated_data:
    node = reimburse_human_data.nodes_by_key[generated_data[key]['dialog_node_key']]
    cleaned_data[key] = generated_data[key]
    for i in range (1, NUM_QUESTIONS+1):
        cleaned_data[key]['text'] = cleaned_data[key]['text'].replace(f"{i}.", "").strip()
    cleaned_data[key]["node_text"] = node.text
    cleaned_data[key]["node_type"] = node.node_type.value

with open("../../../resources/en/reimburse/generated/gpt4o/train_questions_v1.json", "w") as f:
    json.dump(cleaned_data, f)

In [60]:
# TODO Generation Stats
avg = {}
for key in generated_data:
    node = reimburse_human_data.nodes_by_key[generated_data[key]['dialog_node_key']]
    if not node.key in avg:
        avg[node.key] = 0
    avg[node.key] += 1
values = avg.values()
print(sum(values)/len(values))

80.2125


In [61]:
print(avg)

{16351700401033947: 91, 16351710117855351: 85, 16351719713796576: 77, 16351756063145903: 82, 16354903154347892: 56, 16354905244322603: 95, 16363751039893796: 64, 16363751447148919: 72, 16363751845458600: 89, 16363752201349915: 44, 16363753477187975: 81, 16363754346769013: 98, 16363754793760356: 83, 16363755463439219: 80, 16363756081293233: 89, 16363756478730906: 88, 16363810595542638: 63, 16363811629822245: 28, 16363812426270449: 89, 16363834594338823: 55, 16363834981763431: 82, 16363835964385427: 95, 16365520936325006: 91, 16365521324065600: 41, 16365524053616136: 99, 16365525056464242: 98, 16365525829145685: 93, 16365527610579139: 92, 16365623179506925: 90, 16369653367436420: 83, 16369654237579351: 65, 16369654648166968: 91, 16369662459644892: 93, 16369666626028975: 95, 16369666644048691: 90, 16369666664574541: 92, 16369666762318746: 82, 16369793221368112: 90, 16369794097104001: 85, 16370471520840317: 90, 16370471522950468: 95, 16370471523750267: 73, 16370481846790432: 91, 1637048353

# V3

In [25]:
def prompt_v3(node_text: str, num_questions: int):
    return f"""Generate {num_questions} questions about the given facts: '{node_text}'."""

def prompt_v3_ner(answer_text: str, ner: str, num_questions: int):
    return f"""Generate {num_questions} questions about the entity '{ner}' from the fact: '{answer_text}'."""

def api_completion_v3(node_text: str, num_questions: int, temperature: float, seed: int):
    prompt = generate_prompt(system=system, user=prompt_v3(node_text=node_text, num_questions=num_questions))
    output = generate_output(prompt=prompt, temperature=temperature, seed=seed)
    return parse_output(output)

def api_completion_v3_ner(answer_text: str, ner: str, num_questions: int, temperature: float, seed: int):
    prompt = generate_prompt(system=system, user=prompt_v3_ner(answer_text=answer_text, ner=ner, num_questions=num_questions))
    output = generate_output(prompt=prompt, temperature=temperature, seed=seed)
    return parse_output(output)
 

In [26]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import stanza
nlp = stanza.Pipeline('en', processors='tokenize,ner', device="cuda:0")

2024-08-07 13:22:35 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2024-08-07 13:22:35 INFO: Downloaded file to /home/users2/vaethdk/stanza_resources/resources.json
2024-08-07 13:22:35 WARNING: Language en package default expects mwt, which has been added
2024-08-07 13:22:36 INFO: Loading these models for language: en (English):
| Processor | Package                   |
-----------------------------------------
| tokenize  | combined                  |
| mwt       | combined                  |
| ner       | ontonotes-ww-multi_charlm |

2024-08-07 13:22:36 INFO: Using device: cuda:0
2024-08-07 13:22:36 INFO: Loading: tokenize
2024-08-07 13:22:36 INFO: Loading: mwt
2024-08-07 13:22:36 INFO: Loading: ner
2024-08-07 13:22:37 INFO: Done loading processors!


In [21]:
from typing import List, Tuple

def extract_ner_sentences(node: DialogNode) -> List[Tuple[str, str]]:
    """
    Extract all sentences from node text that mention NER's.
    Returns them as a list of tuples, where each tuple contains
        1. the name of the entity
        2. the sentence containing that entity
    """
    results = []
    context = nlp(node.text)
    entities = context.ents
    for entity in entities:
        start_idx = entity.start_char
        end_idx = entity.end_char
        # expand start index to beginning of sentence
        while start_idx > 0 and node.text[start_idx-1] != ".":
            start_idx -= 1
        # expand end index to end of sentence
        while end_idx < len(node.text) and node.text[end_idx-1] != ".":
            end_idx += 1
        results.append((entity.text, node.text[start_idx:end_idx]))
    return results


In [30]:
NUM_QUESTIONS_PER_NER = 100
NUM_QUESTIONS_PER_NODE = 100
TEMPERATURE = 0.7
SEED = 42

system = """You are a truthful assistant, generating diverse FAQ-style questions given some facts.
The generated questions should be answerable using the given facts only, without additional knowledge.
The questions should also be short and human-like.
Try to vary the amount of information between questions.
Use casual language.
Output only the generated paraphrases, separating each paraphrase with a <br> tag."""

generated_data = {}
generated_data_unnumbered = {}

for node in tqdm(reimburse_human_data.nodes_by_type[NodeType.INFO]):
    # use dict indexed by generated text to filter out duplicates
    all_generations = {}
    
    # extract NERs
    named_entities = extract_ner_sentences(node)
    # print(named_entities)

    # Generate questions with NER sentences only, make asking about NER a requirement
    for entity, sentence in named_entities:
        questions = api_completion_v3_ner(node.text, entity, NUM_QUESTIONS_PER_NER, temperature=TEMPERATURE, seed=SEED)

        for question in questions:
            key = str(time.time()).replace(".", "")
            all_generations[key] = {
                "key": key,
                "context": "ner",
                "entity": entity,
                "dialog_node_key": node.key,
                "node_text": node.text,
                "text": question
            }
        # print("- ENTITY", entity)
        # print(questions)
            
    # Generate questions with whole context
    questions = api_completion_v3(node.text, NUM_QUESTIONS_PER_NODE, temperature=TEMPERATURE, seed=SEED)

    for question in questions:
            key = str(time.time()).replace(".", "")
            all_generations[key] = {
                "key": key,
                "context": "node",
                "dialog_node_key": node.key,
                "node_text": node.text,
                "text": question
            }
    
    # filter out duplicates
    uniques = set()
    for question_key in all_generations:
        question = all_generations[question_key]
        if question['text'].lower() in uniques:
            continue # skip duplicate
        else:
            uniques.add(question['text'].lower())
            generated_data[question['key']] = question

  1%|▏         | 1/80 [00:22<29:10, 22.16s/it]

SHORT: 
 - duplicates: 9
 - duplicates: 1


  2%|▎         | 2/80 [01:29<1:03:09, 48.58s/it]

SHORT: 
 - duplicates: 2
SHORT: 
 - duplicates: 44


  4%|▍         | 3/80 [02:24<1:06:29, 51.81s/it]

...

SHORT: 
 - duplicates: 1


 99%|█████████▉| 79/80 [54:09<00:33, 33.96s/it]

SHORT: 
 - duplicates: 0
 - duplicates: 1
SHORT: 
 - duplicates: 1


100%|██████████| 80/80 [55:08<00:00, 41.35s/it]

SHORT: 
 - duplicates: 0


In [33]:
import json

human_data_test = ReimburseGraphDataset('en/reimburse/test_graph.json', 'en/reimburse/test_answers.json', False, augmentation=DataAugmentationLevel.NONE, resource_dir="../../../resources")

overlaps = 0
cleaned_data = {}
for key in generated_data:
    node = reimburse_human_data.nodes_by_key[generated_data[key]['dialog_node_key']]
    test_node = human_data_test.nodes_by_key[generated_data[key]['dialog_node_key']]
    test_questions = set([q.text.strip().lower() for q in test_node.questions])
    if not generated_data[key]['text'].lower() in test_questions:
        # filter out overlap with test set!
        cleaned_data[key] = generated_data[key]
        cleaned_data[key]["node_text"] = node.text
        cleaned_data[key]["node_type"] = node.node_type.value
    else:
        overlaps += 1

print("REMOVED OVERLAP WITH TEST SET:", overlaps, "instances")

with open("../../../resources/en/reimburse/generated/gpt4o/train_questions_v3.json", "w") as f:
    json.dump(cleaned_data, f)


- not using synonyms
===== Dataset Statistics =====
- files:  en/reimburse/test_graph.json en/reimburse/test_answers.json
- synonyms: False
- depth: 20  - degree: 13
- answers: 81
- questions: 173
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  4
- answer limit: 0  - maximum loaded:  1
REMOVED OVERLAP WITH TEST SET: 2 instances
